### Boilerplate code - llm initiation

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

load_dotenv("./env")

google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")

google_llm = ChatGoogleGenerativeAI(
    temperature=0, 
    model="gemini-2.5-flash", 
    api_key=google_api_key,
    max_tokens=200
)

openai_llm = ChatOpenAI(
    temperature=0, 
    model="gpt-5-mini", 
    api_key=openai_api_key
)

ollama_llm = ChatOllama(
    model="llama3.1:latest",
    temperature=0
)

# ollama_llm.invoke("I am from Mars").content

In [ ]:
from langchain_openai import ChatOpenAI

def multiply(a: int, b: int) -> int:
    """Multiply a and b.

    Args:
        a: first int
        b: second int
    """
    return a * b

# This will be a tool
def add(a: int, b: int) -> int:
    """Adds a and b.

    Args:
        a: first int
        b: second int
    """
    return a + b

def divide(a: int, b: int) -> float:
    """Divide a by b.

    Args:
        a: first int
        b: second int
    """
    return a / b

tools = [add, multiply, divide]
llm_with_tools = ollama_llm.bind_tools(tools)

In [ ]:
from IPython.display import Image, display

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import MessagesState
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# System message
sys_msg = SystemMessage(content="You are a helpful assistant tasked with performing arithmetic on a set of inputs.")

# Node
def assistant(state: MessagesState):
   return {"messages": [llm_with_tools.invoke([sys_msg] + state["messages"])]}

# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine the control flow
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
graph = builder.compile(checkpointer=MemorySaver())

# Show
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# Input
initial_input = {"messages": HumanMessage(content="Multiply 2 and 3")}

# Thread
thread = {"configurable": {"thread_id": "1"}}

# Run the graph until the first interruption
for event in graph.stream(initial_input, thread, stream_mode="values"):
    event['messages'][-1].pretty_print()

###  Browsing the state history of our agent. (.get_state & .get_state_history)

In [ ]:
graph.get_state({'configurable': {'thread_id': '1'}})

In [ ]:
all_states = [s for s in graph.get_state_history(thread)]
len(all_states)

### States are updated in order i.e Newest to the oldest
- ##### We chose [-2] as it is the meaningful first state. 
- ##### [-1] is Langgraph's empty/blank state at graph initialization

In [70]:
to_replay = all_states[-2]
to_replay

StateSnapshot(values={'messages': [HumanMessage(content='Multiply 2 and 3', additional_kwargs={}, response_metadata={}, id='7282e0ec-b1cb-4155-bd55-4e4546f92496')]}, next=('assistant',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1215b3-b914-6e02-8000-5a19bdde4d80'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-03-16T17:11:55.031491+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1215b3-b912-656c-bfff-79119e01e099'}}, tasks=(PregelTask(id='7e077398-f8ee-a3f4-0428-199b079dcb01', name='assistant', path=('__pregel_pull', 'assistant'), error=None, interrupts=(), state=None, result={'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1:latest', 'created_at': '2026-03-16T17:11:59.076166Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4042555208, 'load_duration': 2856514417, 'prompt_eval_count': 278, 'prompt_eval_duration': 

In [71]:
to_replay.values

{'messages': [HumanMessage(content='Multiply 2 and 3', additional_kwargs={}, response_metadata={}, id='7282e0ec-b1cb-4155-bd55-4e4546f92496')]}

In [72]:
to_replay.next

('assistant',)

In [73]:
to_replay.config

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1215b3-b914-6e02-8000-5a19bdde4d80'}}

##### It just replay's(rerun) from the if we invoke again with config

In [74]:
for event in graph.stream(None, to_replay.config, stream_mode="values"):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

Multiply 2 and 3
================================== Ai Message ==================================
Tool Calls:
  multiply (747edb6e-ed46-4e31-9714-37fbc2e28ea5)
 Call ID: 747edb6e-ed46-4e31-9714-37fbc2e28ea5
  Args:
    a: 2
    b: 3
================================= Tool Message =================================
Name: multiply

6
================================== Ai Message ==================================

The result of multiplying 2 and 3 is 6.


### Forking a state (to replay with different value)

In [83]:
forked_state = all_states[-2]
forked_state.config

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1215b3-b914-6e02-8000-5a19bdde4d80'}}

In [91]:
forked_config = graph.update_state(
    forked_state.config,
    {"messages": [HumanMessage(content="Multiply 5 and 5", id=forked_state.values['messages'][0].id)]}
)
forked_config

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f12166a-b3e6-6842-8001-aee871e71b65'}}

In [92]:
all_states = [state for state in graph.get_state_history(thread) ]
all_states[0].values["messages"]

[HumanMessage(content='Multiply 5 and 5', additional_kwargs={}, response_metadata={}, id='7282e0ec-b1cb-4155-bd55-4e4546f92496')]

In [93]:
graph.get_state({'configurable': {'thread_id': '1'}})

StateSnapshot(values={'messages': [HumanMessage(content='Multiply 5 and 5', additional_kwargs={}, response_metadata={}, id='7282e0ec-b1cb-4155-bd55-4e4546f92496')]}, next=('assistant',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f12166a-b3e6-6842-8001-aee871e71b65'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-03-16T18:33:46.857063+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1215b3-b914-6e02-8000-5a19bdde4d80'}}, tasks=(PregelTask(id='cd028e33-7da5-7695-2469-7eb79147ecc1', name='assistant', path=('__pregel_pull', 'assistant'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [94]:
for event in graph.stream(None, forked_config, stream_mode="values"):
    event['messages'][-1].pretty_print()

================================ Human Message =================================

Multiply 5 and 5
================================== Ai Message ==================================
Tool Calls:
  multiply (604f85bc-ad99-459f-92e7-73861cfb2852)
 Call ID: 604f85bc-ad99-459f-92e7-73861cfb2852
  Args:
    a: 5
    b: 5
================================= Tool Message =================================
Name: multiply

25
================================== Ai Message ==================================

The result of multiplying 5 and 5 is 25.
